# Bangla & Code-Mixed Smishing Detection (Phases 2–5: GPU Pipeline)

Designed for **Google Colab (T4 / L4 / A100)** within a ~35–50 Compute Unit budget (out of 200).

### What this notebook runs
1. **Setup & Upload/Mount:** Loads `data/processed/unified_3class_dedup.parquet` and `banglaphish2026_dedup.parquet` produced locally by Phase 1–2.
2. **Phase 2 (Encoder Baselines):** Fair 5-fold deduplicated grouped CV for `csebuetnlp/banglabert`, `google/muril-base-cased`, `xlm-roberta-base`, `distilbert-base-multilingual-cased`, and `google/mobilebert-uncased`.
3. **Phase 3 (Shortcut Resistance):** Evaluates models on `<URL>/<PHONE>/<USSD>`-masked text, adversarial ZWJ/homoglyph text, Banglish/CodeMix subsets, length-matched (`40–120` chars) subsets, and cross-corpus OOD (`BanglaPhish-2026` SMS).
4. **Phase 4 (On-Device Lightweight + INT8 Quantization):** Exports best compact encoder (`distilbert-base-multilingual-cased` or distilled student) to ONNX + INT8 dynamic quantization, measuring size (MB), RAM delta, p50/p95 CPU latency, and Macro-F1.
5. **Phase 5 (Selective Quantized SLM Explainer):** Runs 4-bit quantized 1B–1.5B SLM (`Qwen/Qwen2.5-1.5B-Instruct` or `google/gemma-3-1b-it`) *only* on borderline/Smishing predictions grounded with extracted structural cues, exporting samples for human faithfulness scoring.

In [ ]:
#@title 1. Install GPU & ONNX Dependencies
!pip -q install -U transformers datasets accelerate scikit-learn onnxruntime onnx bitsandbytes sentencepiece protobuf

In [ ]:
#@title 2. Mount Google Drive or Upload Processed Parquet Files
import os, torch
from pathlib import Path
print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Option A: If you uploaded data/processed/*.parquet to Drive:
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/bangla_smish/data/processed')

# Option B: Direct upload to Colab /content:
DATA_DIR = Path('/content/data/processed')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path('/content/reports_gpu')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Place unified_3class_dedup.parquet and banglaphish2026_dedup.parquet into:', DATA_DIR)

In [ ]:
#@title 3. Load Deduplicated Datasets & Verify Zero Split Leakage
import pandas as pd
import numpy as np

CANONICAL_LABELS = ('Normal', 'Promotional', 'Smishing')
LABEL_TO_IDX = {k: i for i, k in enumerate(CANONICAL_LABELS)}

df = pd.read_parquet(DATA_DIR / 'unified_3class_dedup.parquet')
bp_df = pd.read_parquet(DATA_DIR / 'banglaphish2026_dedup.parquet')
print('Unified 3-class dedup:', df.shape, df['label_3class'].value_counts().to_dict())
print('BanglaPhish-2026 dedup:', bp_df.shape, bp_df['label_3class'].value_counts().to_dict())
assert df.groupby('dup_cluster_id')['fold'].nunique().max() == 1, 'Cluster leakage detected!'

In [ ]:
#@title 4. Metrics & Bootstrap Helpers (Matches Local Phase 2-3 Protocol)
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import binomtest

def smishing_recall_at_fpr(y_true, probs, smish_idx=2, normal_idx=0, max_fpr=0.01):
    is_smish = y_true == smish_idx
    is_norm = y_true == normal_idx
    scores = probs[:, smish_idx]
    norm_scores = np.sort(scores[is_norm])[::-1]
    allowed_fp = int(np.floor(max_fpr * len(norm_scores)))
    thresh = float(norm_scores[0] + 1e-9) if allowed_fp == 0 else float(norm_scores[allowed_fp - 1])
    preds = scores >= thresh
    return round(float((preds & is_smish).sum() / max(1, is_smish.sum()) * 100), 2)

def eval_bundle(y_true, probs, n_boot=1000, seed=42):
    y_pred = probs.argmax(axis=1)
    rng = np.random.default_rng(seed)
    f1s = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        f1s.append(f1_score(y_true[idx], y_pred[idx], average='macro', zero_division=0))
    lo, hi = np.percentile(f1s, [2.5, 97.5])
    rep = classification_report(y_true, y_pred, target_names=list(CANONICAL_LABELS), output_dict=True, zero_division=0)
    norm_mask = y_true == 0
    fpr = float(((y_pred == 2) & norm_mask).sum() / max(1, norm_mask.sum()) * 100)
    return {
        'n': int(len(y_true)),
        'acc': round(accuracy_score(y_true, y_pred) * 100, 2),
        'macro_f1': round(f1_score(y_true, y_pred, average='macro', zero_division=0) * 100, 2),
        'macro_f1_ci95': [round(float(lo)*100, 2), round(float(hi)*100, 2)],
        'smish_recall': round(rep['Smishing']['recall'] * 100, 2),
        'normal_as_smish_fpr': round(fpr, 2),
        'recall_at_1pct_fpr': smishing_recall_at_fpr(y_true, probs),
    }

In [ ]:
#@title 5. Phase 2 & 3: Train & Evaluate Encoders Across 5-Fold Dedup CV + Stress Sets
import json, time
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODELS = [
    'csebuetnlp/banglabert',
    'google/muril-base-cased',
    'xlm-roberta-base',
    'distilbert-base-multilingual-cased',
    'google/mobilebert-uncased',
]

def train_eval_encoder(model_name, text_col='text', folds_to_run=(0,1,2,3,4), epochs=3, batch_size=32, lr=3e-5, max_len=128):
    tok = AutoTokenizer.from_pretrained(model_name)
    y_all = df['label_3class'].map(LABEL_TO_IDX).to_numpy(dtype=int)
    oof_probs = np.zeros((len(df), 3), dtype=float)
    adv_probs = np.zeros((len(df), 3), dtype=float)
    mask_probs = np.zeros((len(df), 3), dtype=float)

    def tok_fn(batch):
        return tok(batch['txt'], truncation=True, padding='max_length', max_length=max_len)

    for f in folds_to_run:
        tr_mask = (df['fold'] != f).values
        te_mask = (df['fold'] == f).values
        ds_tr = Dataset.from_dict({'txt': df.loc[tr_mask, text_col].tolist(), 'label': y_all[tr_mask].tolist()}).map(tok_fn, batched=True)
        ds_te = Dataset.from_dict({'txt': df.loc[te_mask, text_col].tolist(), 'label': y_all[te_mask].tolist()}).map(tok_fn, batched=True)
        ds_adv = Dataset.from_dict({'txt': df.loc[te_mask, 'text_adv'].tolist(), 'label': y_all[te_mask].tolist()}).map(tok_fn, batched=True)
        ds_msk = Dataset.from_dict({'txt': df.loc[te_mask, 'text_masked'].tolist(), 'label': y_all[te_mask].tolist()}).map(tok_fn, batched=True)

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        args = TrainingArguments(
            output_dir=f'/content/ckpts/{model_name.replace("/", "_")}_f{f}',
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=64,
            learning_rate=lr,
            fp16=torch.cuda.is_available(),
            save_strategy='no',
            logging_strategy='epoch',
            report_to='none',
            seed=42 + f,
        )
        trainer = Trainer(model=model, args=args, train_dataset=ds_tr)
        trainer.train()

        for ds_eval, out_arr in [(ds_te, oof_probs), (ds_adv, adv_probs), (ds_msk, mask_probs)]:
            logits = trainer.predict(ds_eval).predictions
            probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
            out_arr[te_mask] = probs

    # Run on folds_to_run indices
    eval_idx = df['fold'].isin(folds_to_run).values
    lm_idx = eval_idx & (df['char_len'] >= 40).values & (df['char_len'] <= 120).values
    bm_idx = eval_idx & df['source_lang_tag'].isin(['Banglish', 'CodeMix']).values
    res = {
        'clean_cv': eval_bundle(y_all[eval_idx], oof_probs[eval_idx]),
        'masked_eval': eval_bundle(y_all[eval_idx], mask_probs[eval_idx]),
        'adv_zwj_homoglyph': eval_bundle(y_all[eval_idx], adv_probs[eval_idx]),
        'length_matched_40_120': eval_bundle(y_all[lm_idx], oof_probs[lm_idx]),
        'banglish_codemix': eval_bundle(y_all[bm_idx], oof_probs[bm_idx]),
    }
    return res

# Quick pilot on fold=(0,) first (~5 min total), then set folds_to_run=(0,1,2,3,4) for paper table!
gpu_results = {}
for m in MODELS:
    print('=== Running:', m, '===')
    gpu_results[m] = train_eval_encoder(m, text_col='text', folds_to_run=(0,))
    print(json.dumps(gpu_results[m], indent=2))
with open(OUT_DIR / 'phase2_3_encoders.json', 'w') as f:
    json.dump(gpu_results, f, indent=2)

In [ ]:
#@title 6. Phase 4: Export Compact Student to ONNX + INT8 Quantization & Profile CPU Latency/RAM
import os, psutil, time
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Example: export best fold-0 checkpoint of distilbert-base-multilingual-cased (or your distilled student)
STUDENT_MODEL = 'distilbert-base-multilingual-cased'
tok = AutoTokenizer.from_pretrained(STUDENT_MODEL)
pt_model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL, num_labels=3).eval()
dummy = tok('আপনার বিকাশ অ্যাকাউন্ট বন্ধ হয়ে যাবে', return_tensors='pt', padding='max_length', max_length=128)

onnx_fp32 = '/content/student_fp32.onnx'
onnx_int8 = '/content/student_int8.onnx'
torch.onnx.export(
    pt_model, (dummy['input_ids'], dummy['attention_mask']), onnx_fp32,
    input_names=['input_ids', 'attention_mask'], output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'}},
    opset_version=14
)
quantize_dynamic(onnx_fp32, onnx_int8, weight_type=QuantType.QInt8)
print('FP32 MB:', round(os.path.getsize(onnx_fp32)/1e6, 2), '| INT8 MB:', round(os.path.getsize(onnx_int8)/1e6, 2))

sess_opts = ort.SessionOptions()
sess_opts.intra_op_num_threads = 4
sess = ort.InferenceSession(onnx_int8, sess_options=sess_opts, providers=['CPUExecutionProvider'])
sample_texts = df['text'].head(200).tolist()
latencies_ms = []
for t in sample_texts:
    inp = tok(t, return_tensors='np', truncation=True, max_length=128)
    t0 = time.perf_counter()
    sess.run(None, {'input_ids': inp['input_ids'], 'attention_mask': inp['attention_mask']})
    latencies_ms.append((time.perf_counter() - t0) * 1000)
print('INT8 CPU Latency p50 (ms):', round(float(np.percentile(latencies_ms, 50)), 2),
      '| p95 (ms):', round(float(np.percentile(latencies_ms, 95)), 2))

In [ ]:
#@title 7. Phase 5: Selective 4-bit SLM Explainer (Cue-Grounded Bangla Safety Rationales)
import re
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

SLM_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # Alternative: 'google/gemma-3-1b-it'
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
slm_tok = AutoTokenizer.from_pretrained(SLM_ID)
slm = AutoModelForCausalLM.from_pretrained(SLM_ID, quantization_config=bnb_cfg, device_map='auto')

def extract_cues(text):
    urls = re.findall(r'https?://\S+|(?:bit\.ly|cutt\.ly|t\.me)/\S+', text, re.I)
    phones = re.findall(r'(?:\+?880[-\s]?1[3-9]\d{8}|\b01[3-9]\d{8}\b)', text)
    brands = [b for b in ['bKash', 'Nagad', 'Rocket', 'বিকাশ', 'নগদ', 'রকেট', 'সোনালী ব্যাংক', 'OTP', 'PIN'] if b.lower() in text.lower()]
    return {'urls': urls, 'phones': phones, 'brands': brands}

def explain_smishing(sms_text):
    cues = extract_cues(sms_text)
    prompt = (
        f'SMS: {sms_text}\n'
        f'Detected cues: {cues}\n'
        'Write a 2-sentence warning in simple Bangla explaining why this SMS is risky. Only mention cues present above.'
    )
    msgs = [{'role': 'user', 'content': prompt}]
    inp = slm_tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(slm.device)
    out = slm.generate(inp, max_new_tokens=90, do_sample=False)
    return slm_tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True)

for s in df[df['label_3class'] == 'Smishing']['text'].head(3):
    print('SMS:', s)
    print('Explanation:', explain_smishing(s))
    print('-'*60)